# Erster Test: Gemini 2.5 Pro (über OpenRouter) mit den sechs Testfragen

Ziel: grob prüfen, ob Gemini 2.5 Pro die Testfragen in `data/synthetic_reports/test_questions.md` sinnvoll beantwortet.

- **Noch keine Suche (Retrieval):** Alle fiktiven Berichte aus `data/synthetic_reports` gehen vollständig in den Prompt. So sieht man, was das Modell kann, bevor die Suche als zusätzliche Fehlerquelle dazukommt.
- **Daten:** Nur die fiktiven Testberichte aus diesem Repository. Anfragen laufen über OpenRouter zu Google.
- **Schlüssel:** Wird aus den Colab-Secrets gelesen (Schlüsselsymbol links), Name `OPENROUTER_API_KEY`.
- **Bewertung:** Antworten danach von Hand mit den Antwortentwürfen und Kriterien in `test_questions.md` vergleichen.

In [21]:
!pip install -q -U google-genai
!git clone -q https://github.com/Julien0049/AI-database.git || git -C AI-database pull -q

fatal: destination path 'AI-database' already exists and is not an empty directory.


In [26]:
import os
from google.colab import userdata
# Wir nutzen das openai-Paket, um OpenRouter anzusprechen
!pip install -q openai
from openai import OpenAI

KEY_NAME = "OPENROUTER_API_KEY"
# Wir wechseln auf Gemini 2.5 Pro über OpenRouter
MODEL = "google/gemini-2.5-pro"

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=userdata.get(KEY_NAME),
)

In [23]:
import pathlib, re

DATA = pathlib.Path("AI-database/data/synthetic_reports")
reports = sorted(DATA.glob("AAR-*.md"))
corpus = "\n\n".join(
    f'<dokument datei="{p.name}">\n{p.read_text()}\n</dokument>' for p in reports
)

# Die Fragen stehen in test_questions.md jeweils als Zitat unter "**Frage**".
questions = re.findall(r"\*\*Frage\*\*\s*\n\s*> (.+)", (DATA / "test_questions.md").read_text())
print(len(reports), "Berichte,", len(questions), "Fragen")

5 Berichte, 6 Fragen


## Antwortregeln

Der Systemprompt übernimmt die bestätigten Antwortregeln aus `AGENTS.md`.

In [24]:
SYSTEM = """Du beantwortest Fragen ausschließlich auf Grundlage der mitgelieferten Berichte.
Die Berichte sind fiktive Testdaten.

Regeln:
1. Verwende nur Aussagen aus den Berichten. Ergänze nichts aus eigenem Wissen.
2. Ergänzende Aussagen aus mehreren Berichten darfst du zu einer klaren Antwort zusammenführen.
3. Wenn Berichte mit vergleichbarem Kontext sich widersprechen, bestimmt die jüngste passende Aussage die Antwort.
   Nenne ältere abweichende Verfahren nicht als gleichrangige Alternativen.
4. Wenn Berichte unterschiedliche Kontexte betreffen oder keine sichere Priorisierung zulassen, behaupte keine Priorität.
5. Bezeichne keine Quelle als gültig oder verbindlich; die Berichte belegen das nicht.
6. Wenn die Berichte die Frage nicht ausreichend beantworten, sag das offen und fülle keine Lücken.

Format: Zuerst die knappe Antwort. Danach eine Zeile "Quellen:" mit Dateiname und Datum der Berichte, die die Antwort stützen."""

In [27]:
import time

results = []
for i, question in enumerate(questions, 1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"{corpus}\n\nFrage: {question}"}
        ]
    )
    answer = response.choices[0].message.content
    results.append({"nr": i, "frage": question, "antwort": answer})
    print(f"## Frage {i}\n{question}\n\n{answer}\n")
    time.sleep(5)  # Abstand wegen der Limits

## Frage 1
Ich nehme an einem dreitägigen Seminar teil. Unsere Gruppe hat ein Materialset gemeinsam genutzt. Wie wird die Rückgabe bestätigt?

Jede teilnehmende Person bestätigt die Rückgabe der von ihr genutzten Bestandteile auf der persönlichen Liste. Zusätzlich reicht die Gruppenvertretung eine Gesamtübersicht des gemeinsam genutzten Sets ein.

Quellen: AAR-2025-01-20-shared-set-returns.md (20.01.2025)

## Frage 2
Bei unserem dreitägigen Seminar fehlt nach der gemeinsamen Nutzung ein Adapter aus dem Materialset. Wie dokumentieren wir die Rückgabe und den fehlenden Adapter?

Bei der Rückgabe des gemeinsam genutzten Sets bestätigt jede teilnehmende Person auf ihrer persönlichen Liste die Rückgabe der von ihr genutzten Bestandteile. Zusätzlich reicht die Gruppenvertretung eine Gesamtübersicht des Sets ein.

Der fehlende Adapter wird mit Set-Kennung, Gegenstandsbezeichnung und Entdeckungsdatum im Abweichungsprotokoll erfasst. Die Materialverwaltung wird noch am selben Tag informiert. Au

## Ergebnisse speichern

Lädt eine JSON-Datei mit Modell, Zeitpunkt, Systemprompt und Antworten herunter. Sie kann zur Auswertung ins Repository übernommen werden.

In [30]:
import json, datetime
from google.colab import files

run = {
    "modell": MODEL,
    "zeitpunkt": datetime.datetime.now().isoformat(timespec="seconds"),
    "systemprompt": SYSTEM,
    "ergebnisse": results,
}
# Ersetzt Schrägstriche im Modellnamen durch Unterstriche, um Pfadfehler zu verhindern
safe_model_name = MODEL.replace("/", "_")
name = f"ergebnisse_{safe_model_name}_{datetime.date.today()}.json"
pathlib.Path(name).write_text(json.dumps(run, ensure_ascii=False, indent=2))
files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>